# Task 1: Exploratory Data Analysis — Financial News Dataset

## Business Objective
Financial news headlines directly influence investor behavior and stock prices.
This analysis explores a large dataset of financial news to uncover publication
patterns, identify key topics, and understand publisher behavior.
These insights will inform sentiment analysis and stock correlation in Task 3.

**Author:** Sosina Ayele  
**Dataset:** Raw analyst ratings — 1.4M financial news headlines

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')
print('Libraries imported successfully')

## 1. Load and Inspect Data

In [ ]:
import os
paths = [
    '../data/raw/raw_analyst_ratings.csv',
    r'c:\KAIM\news-sentiment-analysis\data\raw\raw_analyst_ratings.csv',
]
df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded from: {path}')
        break

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
df.info()
df.describe(include='all')

## 2. Descriptive Statistics
Analyzing headline character length and word count to understand the dataset's textual properties.

In [ ]:
df['headline_length'] = df['headline'].astype(str).apply(len)
df['word_count'] = df['headline'].astype(str).apply(lambda x: len(x.split()))

print('=== Headline Character Length ===')
print(df['headline_length'].describe().round(2))
print('\n=== Word Count ===')
print(df['word_count'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['headline_length'], bins=40, ax=axes[0], color='steelblue')
axes[0].axvline(df['headline_length'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['headline_length'].mean():.1f}")
axes[0].set_title('Headline Length Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.histplot(df['word_count'], bins=30, ax=axes[1], color='coral')
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['word_count'].mean():.1f}")
axes[1].set_title('Word Count Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.suptitle('Descriptive Statistics — Headline Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('headline_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 1 saved!')

## 3. Publisher Analysis
Identifying the most active publishers and extracting domains from email-format publisher names.

In [ ]:
publisher_counts = df['publisher'].value_counts().head(15)
print('=== Top 15 Publishers ===')
print(publisher_counts)

df['publisher_domain'] = df['publisher'].astype(str).str.extract(r'@([\w\.-]+)')
print('\n=== Top Email Domains ===')
print(df['publisher_domain'].value_counts().head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

publisher_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('viridis', len(publisher_counts)))
axes[0].set_title('Top 15 Most Active Publishers', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Article Count')

domain_counts = df['publisher_domain'].value_counts().head(10)
domain_counts.plot(kind='barh', ax=axes[1], color=sns.color_palette('plasma', len(domain_counts)))
axes[1].set_title('Top 10 Publisher Domains', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Article Count')

plt.suptitle('Publisher Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('publisher_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 2 saved!')

## 4. Time Series Analysis
Analyzing publication frequency over time to identify market event spikes and publishing patterns.

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
df = df.dropna(subset=['date'])
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()

daily_news = df.groupby(df['date'].dt.date).size()
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'\nArticles per day stats:\n{daily_news.describe().round(2)}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(daily_news.index, daily_news.values, color='steelblue', linewidth=0.8)
axes[0,0].fill_between(daily_news.index, daily_news.values, alpha=0.3, color='steelblue')
axes[0,0].set_title('Daily News Volume', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('Date')
axes[0,0].set_ylabel('Articles Published')

monthly = df.groupby(df['date'].dt.to_period('M')).size()
axes[0,1].bar(range(len(monthly)), monthly.values, color='coral')
axes[0,1].set_title('Monthly News Volume', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Month Index')
axes[0,1].set_ylabel('Articles')

hour_counts = df['hour'].value_counts().sort_index()
axes[1,0].bar(hour_counts.index, hour_counts.values, color='steelblue')
axes[1,0].axvspan(13, 20, alpha=0.2, color='green', label='US Market Hours')
axes[1,0].set_title('Publishing Frequency by Hour (UTC)', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('Hour')
axes[1,0].set_ylabel('Count')
axes[1,0].legend()

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
axes[1,1].bar(day_counts.index, day_counts.values, color='coral')
axes[1,1].set_title('Articles by Day of Week', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Day')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Time Series Analysis — Publication Patterns', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('time_series_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 3 saved!')

## 5. Text Analysis — TF-IDF Keyword Extraction
Using TF-IDF to extract the most significant keywords and bigrams from financial news headlines.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_features=25)
X = tfidf.fit_transform(df['headline'].astype(str))
scores = pd.DataFrame({
    'keyword': tfidf.get_feature_names_out(),
    'score': X.sum(axis=0).A1
}).sort_values('score', ascending=False)

print('=== Top 15 TF-IDF Keywords ===')
print(scores.head(15))

vectorizer = CountVectorizer(stop_words='english', ngram_range=(2,2), max_features=20)
X2 = vectorizer.fit_transform(df['headline'].astype(str))
phrases = pd.DataFrame({
    'phrase': vectorizer.get_feature_names_out(),
    'count': X2.sum(axis=0).A1
}).sort_values('count', ascending=False)

print('\n=== Top Bigrams (2-word phrases) ===')
print(phrases.head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_scores = scores.head(15)
axes[0].barh(top_scores['keyword'][::-1], top_scores['score'][::-1],
             color=sns.color_palette('magma', len(top_scores)))
axes[0].set_title('Top 15 Keywords — TF-IDF', fontsize=13, fontweight='bold')
axes[0].set_xlabel('TF-IDF Score')

top_phrases = phrases.head(15)
axes[1].barh(top_phrases['phrase'][::-1], top_phrases['count'][::-1],
             color=sns.color_palette('viridis', len(top_phrases)))
axes[1].set_title('Top 15 Bigrams — Count Vectorizer', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')

plt.suptitle('Text Analysis — Keyword & Topic Extraction', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('tfidf_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot 4 saved!')

## 6. Key Insights & Summary

### Findings:
1. **Headline Length:** Average ~73 characters — short, information-dense financial reporting
2. **Publisher Concentration:** Small number of publishers dominate the dataset
3. **Time Patterns:** Most news published during US market hours (13:00–20:00 UTC)
4. **Weekday Pattern:** Significantly fewer articles published on weekends
5. **Top Keywords:** 'price target', 'earnings', 'upgrade', 'downgrade' dominate headlines
6. **Bigrams:** 'price target' and 'earnings beat' are the most common 2-word phrases

### Next Steps (Task 2):
- Load historical stock price data for AAPL, AMZN, GOOG, META, NVDA
- Compute SMA, EMA, RSI, MACD technical indicators using `ta` library
- Visualize price action with indicators for each stock